# 🏥 MediSort AI — Deep Learning & CNN-Based Medical Waste Classification
**Tech Stack:** Python | CNN | TensorFlow | Keras | Scikit-learn | OpenCV | MobileNetV2 | Statistical Evaluation

### Project Highlights:
- **Transfer Learning Architecture:** Pretrained **MobileNetV2** backbone with custom classification head for robust biomedical waste categorization.
- **WHO Biomedical Waste Segregation:** Multi-class classification across 5 standard WHO waste categories:
  - 🟡 **Infectious Waste** (Yellow Bin: gloves, masks, gauze, swabs, bandages)
  - ⚪ **Sharps Waste** (White/Translucent Puncture-Proof Container: needles, syringes, scalpels, blades)
  - 🟤 **Pharmaceutical Waste** (Brown Bin: capsules, pills, reagents, medicines)
  - 🔴 **Plastic Medical Waste** (Red Bin: IV bags, plastic bottles, tubing, pipettes)
  - 🔵 **General & Glass Waste** (Blue/Black Bin: glass vials, ampoules, clean packaging)
- **OpenCV Feature Pipeline:** Adaptive Histogram Equalization (CLAHE), Bilateral Edge Filtering, and Contour-Based ROI Bounding Box Localization.
- **Rigorous Statistical Evaluation:** Accuracy, Precision, Recall, F1-Score, and Confusion Matrix across all waste categories.

In [ ]:
# Cell 1: Environment Imports & Hardware Verification
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import zipfile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow Version:", tf.__version__)
print("OpenCV Version    :", cv2.__version__)
print("Available GPUs    :", tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 2: Dataset Acquisition & Extraction (Cross-Platform / Colab Compatible)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Locate dataset directory
candidates = [
    "./MedBin_dataset.v7i.multiclass",
    "../MedBin_dataset.v7i.multiclass",
    "/content/MedBin_dataset.v7i.multiclass",
    r"C:\Users\admin\Downloads\MedBin_dataset.v7i.multiclass"
]

DATASET_DIR = None
for c in candidates:
    if os.path.exists(c) and os.path.exists(os.path.join(c, "train")):
        DATASET_DIR = os.path.abspath(c)
        break

if DATASET_DIR is None:
    # If dataset folder doesn't exist, extract zip
    zip_candidates = ["MedBin_dataset.v7i.multiclass.zip", "/content/MedBin_dataset.v7i.multiclass.zip"]
    for z in zip_candidates:
        if os.path.exists(z):
            print(f"Extracting {z}...")
            with zipfile.ZipFile(z, 'r') as zip_ref:
                extract_target = "/content/" if IN_COLAB else "."
                zip_ref.extractall(extract_target)
            DATASET_DIR = os.path.abspath(os.path.join(extract_target, "MedBin_dataset.v7i.multiclass"))
            break

print("Using Dataset Directory:", DATASET_DIR)

In [ ]:
# Cell 3: Directory Structure & Path Validation
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VALID_DIR = os.path.join(DATASET_DIR, "valid")
TEST_DIR = os.path.join(DATASET_DIR, "test")

print("Train Dir:", TRAIN_DIR)
print("Valid Dir:", VALID_DIR)
print("Test Dir :", TEST_DIR)

print("\n--- Verified Class Subdirectories in Train ---")
train_classes = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
for c in sorted(train_classes):
    count = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    print(f"  • {c:<25}: {count:>4} images")

In [ ]:
# Cell 4: Exploratory Data Analysis (EDA) — Class Distribution Visualization
splits_data = []
for split_name, sdir in [('Train', TRAIN_DIR), ('Valid', VALID_DIR), ('Test', TEST_DIR)]:
    if os.path.exists(sdir):
        for c in train_classes:
            cdir = os.path.join(sdir, c)
            count = len(os.listdir(cdir)) if os.path.exists(cdir) else 0
            splits_data.append({'Split': split_name, 'Category': c, 'Count': count})

df_eda = pd.DataFrame(splits_data)

plt.figure(figsize=(12, 5))
palette = {'Train': '#2ecc71', 'Valid': '#3498db', 'Test': '#e74c3c'}
sns.barplot(data=df_eda, x='Category', y='Count', hue='Split', palette=palette)
plt.title('MediSort AI — Dataset Class Distribution Across Splits', fontsize=14, fontweight='bold')
plt.xlabel('Biomedical Waste Category', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.legend(title='Dataset Split')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: OpenCV Preprocessing & Computer Vision Feature Extraction
def opencv_feature_extraction(image_path):
    """
    OpenCV Computer Vision Pipeline:
    1. Color space conversion (BGR -> RGB, BGR -> LAB)
    2. Contrast Limited Adaptive Histogram Equalization (CLAHE) to reveal subtle medical contours
    3. Bilateral Filter for edge-preserving noise suppression
    4. Canny Edge Detection & Morphological Closing
    5. External Contour Localization & ROI Bounding Box Extraction
    """
    bgr = cv2.imread(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    
    # CLAHE
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    enhanced_bgr = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
    enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)
    
    # Edge & Contour ROI
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.bilateralFilter(gray, 9, 75, 75)
    edges = cv2.Canny(blurred, 50, 150)
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    annotated = rgb.copy()
    if contours:
        largest = max(contours, key=cv2.contourArea)
        if cv2.contourArea(largest) > 200:
            x, y, w, h = cv2.boundingRect(largest)
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 3)
            cv2.putText(annotated, "Medical Waste ROI", (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
    return rgb, enhanced_rgb, edges, annotated

# Demonstrate OpenCV feature on sample images
sample_cat = train_classes[0]
sample_file = os.path.join(TRAIN_DIR, sample_cat, os.listdir(os.path.join(TRAIN_DIR, sample_cat))[0])
orig, enhanced, edges, roi_img = opencv_feature_extraction(sample_file)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(orig); axes[0].set_title("1. Original Input"); axes[0].axis("off")
axes[1].imshow(enhanced); axes[1].set_title("2. OpenCV CLAHE"); axes[1].axis("off")
axes[2].imshow(edges, cmap='gray'); axes[2].set_title("3. OpenCV Canny Edges"); axes[2].axis("off")
axes[3].imshow(roi_img); axes[3].set_title("4. OpenCV ROI Localization"); axes[3].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Data Loading Pipeline with tf.keras.utils.image_dataset_from_directory
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print(f"\nClasses Loaded ({num_classes} total):", class_names)

# Optimize performance with prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
valid_ds = valid_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Cell 7: MobileNetV2 Deep Learning Transfer Learning Architecture
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess_input)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="MediSort_MobileNetV2")
model.summary()

In [ ]:
# Cell 8: Model Compilation & Training
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS = 5
print(f"Starting Transfer Learning Training ({EPOCHS} Epochs)...")
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS
)

# Save trained model
model.save("medical_waste_model.h5")
print("Trained model saved to medical_waste_model.h5")

In [ ]:
# Cell 9: Training & Validation Curves (Accuracy and Loss)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy", color="#2ecc71", lw=2)
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", color="#3498db", lw=2)
plt.title("MediSort AI — Accuracy Over Epochs", fontsize=13, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss", color="#e74c3c", lw=2)
plt.plot(history.history["val_loss"], label="Validation Loss", color="#e67e22", lw=2)
plt.title("MediSort AI — Loss Over Epochs", fontsize=13, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Held-Out Test Set Evaluation & Statistical Metrics
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n" + "="*50)
print("         FINAL TEST PERFORMANCE METRICS")
print("="*50)
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1 Score : {f1*100:.2f}%")
print("="*50)
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Cell 11: Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("MediSort AI — Test Confusion Matrix", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Visualizing Predictions on Held-Out Test Samples
for images, labels in test_ds.take(1):
    predictions = model.predict(images, verbose=0)
    plt.figure(figsize=(14, 9))
    num_items = min(9, len(images))
    for i in range(num_items):
        predicted_class = np.argmax(predictions[i])
        confidence = np.max(predictions[i]) * 100
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        actual = class_names[labels[i]]
        predicted = class_names[predicted_class]
        color = "green" if actual == predicted else "red"
        plt.title(f"Actual: {actual}\nPred: {predicted} ({confidence:.1f}%)", color=color, fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13: Interactive OpenCV + MobileNetV2 Medical Waste Predictor Function
WHO_GUIDELINES = {
    'Infectious_Waste': '🟡 Yellow Bin — Incineration / Double Autoclaving',
    'Sharps_Waste': '⚪ White Puncture-Proof Container — Needle destruction & encapsulation',
    'Pharmaceutical_Waste': '🟤 Brown Bin — High-temperature rotary kiln incineration',
    'Plastic_Medical_Waste': '🔴 Red Bin — Autoclave chemical disinfection & recycling',
    'General_and_Glass_Waste': '🔵 Blue/Black Bin — Glass disinfection & clean recycling'
}

def predict_medical_waste(image_path):
    """
    Given any image file:
    1. Extracts OpenCV visual features & ROI
    2. Predicts medical waste category & confidence using MobileNetV2
    3. Provides WHO disposal protocol
    """
    orig, enhanced, edges, roi = opencv_feature_extraction(image_path)
    
    pil_img = Image.open(image_path).convert('RGB').resize((224, 224))
    arr = np.expand_dims(np.array(pil_img, dtype=np.float32), axis=0)
    preds = model.predict(arr, verbose=0)[0]
    
    pred_idx = np.argmax(preds)
    pred_class = class_names[pred_idx]
    conf = preds[pred_idx] * 100
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(orig); ax[0].set_title("Original Image"); ax[0].axis('off')
    ax[1].imshow(roi); ax[1].set_title(f"OpenCV ROI Localization\n{pred_class} ({conf:.1f}%)"); ax[1].axis('off')
    plt.tight_layout(); plt.show()
    
    print(f"\n{'='*55}")
    print(f"Predicted Waste Category: {pred_class}")
    print(f"Confidence Level         : {conf:.2f}%")
    print(f"Recommended Disposal    : {WHO_GUIDELINES.get(pred_class, 'Standard disposal')}")
    print(f"{'='*55}")

# Test on a sample from the test set
sample_test_path = os.path.join(TEST_DIR, class_names[0], os.listdir(os.path.join(TEST_DIR, class_names[0]))[0])
predict_medical_waste(sample_test_path)